In [1]:
# Importing Modules
import pandas as pd
import numpy as np
from src.utils.smiles2morganfp import MorganFP
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/processed/train/ESOL.csv")
esol_test_data = pd.read_csv("data/processed/test/ESOL.csv")

# Generate ESOL FP
esol_train_fp = MorganFP(esol_train_data["smiles"])
esol_train_fp["smiles"] = esol_train_fp.index
esol_train_fp = esol_train_fp.merge(esol_train_data, on="smiles")
esol_test_fp = MorganFP(esol_test_data["smiles"])
esol_test_fp["smiles"] = esol_test_fp.index
esol_test_fp = esol_test_fp.merge(esol_test_data, on="smiles")

######################## DATA-2 ##################################
# Loading RT data
rt_train_data = pd.read_csv("data/processed/train/RT.csv")
rt_test_data = pd.read_csv("data/processed/test/RT.csv")

# Generate RT FP
rt_train_fp = MorganFP(rt_train_data["smiles"])
rt_train_fp["smiles"] = rt_train_fp.index
rt_train_fp = rt_train_fp.merge(rt_train_data, on="smiles")
rt_test_fp = MorganFP(rt_test_data["smiles"])
rt_test_fp["smiles"] = rt_test_fp.index
rt_test_fp = rt_test_fp.merge(rt_test_data, on="smiles")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/processed/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/processed/test/Lipophilicity.csv")

# Generate lipophilicity FP
lipophilicity_train_fp = MorganFP(lipophilicity_train_data["smiles"])
lipophilicity_train_fp["smiles"] = lipophilicity_train_fp.index
lipophilicity_train_fp = lipophilicity_train_fp.merge(lipophilicity_train_data, on="smiles")
lipophilicity_test_fp = MorganFP(lipophilicity_test_data["smiles"])
lipophilicity_test_fp["smiles"] = lipophilicity_test_fp.index
lipophilicity_test_fp = lipophilicity_test_fp.merge(lipophilicity_test_data, on="smiles")

######################## DATA-4 ##################################
# Loading B3DB data
b3db_train_data = pd.read_csv("data/processed/train/B3DB.csv")
b3db_test_data = pd.read_csv("data/processed/test/B3DB.csv")

# Generate B3DB FP
b3db_train_fp = MorganFP(b3db_train_data["smiles"])
b3db_train_fp["smiles"] = b3db_train_fp.index
b3db_train_fp = b3db_train_fp.merge(b3db_train_data, on="smiles")
b3db_test_fp = MorganFP(b3db_test_data["smiles"])
b3db_test_fp["smiles"] = b3db_test_fp.index
b3db_test_fp = b3db_test_fp.merge(b3db_test_data, on="smiles")

In [3]:
# Function to run ML training and testing
def RunML(model, train_data, test_data, dataName, modelName):
    
	# Training data
	X_train, y_train = train_data.drop(["smiles", "target"], axis=1), train_data["target"]
    
    # Test (hold-out) data
	X_test, y_test = test_data.drop(["smiles", "target"], axis=1), test_data["target"]

	# Model training
	model = model.fit(X_train.to_numpy(), y_train)

	# Model testing
	y_pred = model.predict(X_test.to_numpy())

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)
    
    # Bootstrap 95% CI
	rng = np.random.default_rng(42)
	mae_samples = []
	rmse_samples = []

	y_test_arr = np.array(y_test)
	y_pred_arr = np.array(y_pred)

	for _ in range(2000):
		idx = rng.integers(0, len(y_test_arr), len(y_test_arr))
		mae_samples.append(mean_absolute_error(y_test_arr[idx], y_pred_arr[idx]))
		rmse_samples.append(root_mean_squared_error(y_test_arr[idx], y_pred_arr[idx]))

	mae_lower, mae_upper = np.percentile(mae_samples, [2.5, 97.5])
	rmse_lower, rmse_upper = np.percentile(rmse_samples, [2.5, 97.5])

	# Return results
	return pd.DataFrame({ "Data": [dataName], "Model": [modelName],
                          "MAE": [mae], "MAE_lower": [mae_lower],
                          "MAE_upper": [mae_upper], "RMSE": [rmse],
                          "RMSE_lower": [rmse_lower], "RMSE_upper": [rmse_upper]
                        })

In [4]:
# Data dict
datasets = {"ESOL":{"train":esol_train_fp, "test":esol_test_fp},
            "Lipophilicity":{"train":lipophilicity_train_fp, "test":lipophilicity_test_fp},
            "RT":{"train":rt_train_fp, "test":rt_test_fp},
            "B3DB":{"train":b3db_train_fp, "test":b3db_test_fp}}

# Model dict
model_dict = {
    "LR": LinearRegression(), 
    "SVM": SVR(),
    "RF": RandomForestRegressor(random_state=42),
    "XGB": XGBRegressor(random_state=42, objective='reg:squarederror')
}

In [5]:
# List to store results
temp_out = []

# Loop for models
for modelName, model in model_dict.items():
    # Loop for dataset
    for dataName, data in datasets.items():
        # Run Analysis for model and dataset
        params_df = pd.read_csv(f"results/Output_Hyperparameter_Optimization_ML_{dataName}.csv")
        params = eval(params_df[params_df["Model"]== modelName]["Model Params"].to_list()[0])
        model =  model.set_params(**params)
        temp_out.append(RunML(model, data["train"], data["test"], dataName, modelName))

# Final output
ML_out = pd.concat(temp_out, ignore_index=True)
ML_out.to_csv("results/Output_ML_Analysis.csv")
ML_out

,Data,Model,MAE,MAE_lower,MAE_upper,RMSE,RMSE_lower,RMSE_upper
0,ESOL,LR,3.162221,2.755528,3.570210,4.396003,3.868889,4.983004
1,Lipophilicity,LR,1.688580,1.508234,1.876097,2.161665,1.952997,2.381666
2,RT,LR,239.924947,210.523909,273.277637,330.651061,288.663966,374.257820
3,B3DB,LR,0.695221,0.594285,0.803799,1.047856,0.886751,1.207036
4,ESOL,SVM,0.767196,0.678763,0.856510,1.014597,0.900128,1.134246
5,Lipophilicity,SVM,0.759509,0.682826,0.845272,0.956635,0.866739,1.052349
6,RT,SVM,109.353475,97.374924,122.686514,141.025577,126.885860,156.023621
7,B3DB,SVM,0.366541,0.319655,0.416641,0.513486,0.452270,0.574942
8,ESOL,RF,1.175788,1.052866,1.309994,1.498433,1.330664,1.675132
9,Lipophilicity,RF,0.887141,0.793671,0.983987,1.109690,1.004575,1.217392


In [6]:
ML_out = pd.read_csv("results/Output_ML_Analysis.csv")

# Function to plot barplot showing result
def plot_bar(data, target):
    data = data.copy()
    data["err_lower"] = data[target] - data[f"{target}_lower"]
    data["err_upper"] = data[f"{target}_upper"] - data[target]
    sns.set_theme(style="ticks", context="paper")
    model_order = data["Model"].unique()
    g = sns.catplot(
        data=data, kind="bar", x="Model", y=target, hue="Model",
        col="Data", col_wrap=4, sharey=False, height=4, width=0.8, 
        aspect=0.8, dodge=False, errorbar=None, order=model_order)
    for i, ax in enumerate(g.axes.flat):
        col_name = g.col_names[i]
        subdata = data[data["Data"] == col_name].set_index("Model").reindex(model_order).reset_index()
        if not subdata.dropna(subset=[target]).empty:
            ax.errorbar(
                x=np.arange(len(subdata)), 
                y=subdata[target],
                yerr=[subdata["err_lower"], subdata["err_upper"]], 
                fmt="none", capsize=4, linewidth=1, color="black")
    g.set_axis_labels("Model", f"{target}")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=40)
    plt.tight_layout()
    plt.savefig(f"results/ML_Analysis_{target}_Plot.png", dpi=300)
    plt.close()

In [7]:
# Plotting barplot for RMSE
plot_bar(ML_out, "RMSE")
# Plotting barplot for MAE
plot_bar(ML_out, "MAE")